# Le vrai prix de la ville : proximité des stations et prix au m²
**Paris et petite couronne (75, 92, 93, 94), appartements, 2021-2025**

Groupe : BAHAN Karafa Ashley Kevin, TURKI Meriam, NAFTI Mohamed Amin

---

## Fil conducteur
| Étape | Question à laquelle on répond |
|---|---|
| 1-3. Données | D'où viennent les données, sont-elles cohérentes, comment mesure-t-on la distance ? |
| 4. Qualité | Peut-on faire confiance aux effectifs et aux prix ? |
| 5. Géographie | Où sont les prix élevés ? Les stations sont-elles dans les zones chères ? |
| 6. Courbe de décote | Comment le prix varie-t-il avec la distance à la station ? |
| 7. Évolution | La baisse des prix 2021-2025 est-elle la même partout ? |
| 8. Prime de proximité | Quelle part de l'écart brut vient de la localisation (commune) et non de la station ? |
| 9. Tranches du plan et H2 | Y a-t-il une décote à moins de 100 m ? Où est le maximum ? |
| 10. Surface (H3) | Le prix au m² décroît-il avec la surface ? Cela brouille-t-il les courbes ? |
| 11. Robustesse | Le résultat dépend-il des seuils choisis ? |
| 12. Synthèse | Que peut-on affirmer, que reste-t-il à creuser ? |

## Problématique et hypothèses

**Question de recherche.** À caractéristiques comparables, un appartement proche d'une station de métro/RER se vend-il plus cher ? L'effet est-il maximal juste à côté de la station, ou existe-t-il un intervalle de distance « optimal » entre accessibilité et nuisances ?

**Hypothèses (version descriptive).**
- **H1** : à quartier comparable, le prix au m² diminue quand la distance à la station augmente.
- **H2** : la relation n'est pas monotone : les biens à moins de 100 m subissent une décote (bruit, passage) par rapport à ceux situés à 200-400 m.
- **H3** : le prix au m² décroît avec la surface, plus fortement pour les petits logements.

**Comment « à quartier comparable » est traité sans régression.** On compare les biens **à l'intérieur d'une même commune (ou arrondissement)**, puis on fait la moyenne des écarts, pondérée par les effectifs. C'est une **standardisation descriptive** (stratification), pas un modèle à effets fixes.


**Dépendances** : `pip install pandas numpy geopandas matplotlib pyarrow`
Fichier attendu en entrée : `dvf_appartements_stations.parquet`, produit par `code_dvf.py` puis `stations.py` (étape 1).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.1f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ----------------------------- PARAMÈTRES -----------------------------
DEPARTEMENTS = ["75", "92", "93", "94"]
ANNEES = [2021, 2022, 2023, 2024, 2025]
FIG_DIR = Path("figures");  FIG_DIR.mkdir(exist_ok=True)

# Fichier produit par les scripts (code_dvf.py puis stations.py).
NOM_PARQUET = "dvf_appartements_stations.parquet"
NOM_RAPPORT = "rapport_nettoyage.csv"
NOM_GEOJSON_STATIONS = "emplacement_gares_idfm.geojson"   # optionnel : pour la carte

# Analyse
MIN_N = 30        # effectif minimal pour afficher un point d'une courbe
N_BOOT = 200      # rééchantillonnages bootstrap (baisser à 50 pour aller vite)
SEED = 42
rng = np.random.default_rng(SEED)

ZONES = ["Paris", "Petite couronne"]
BORNES_PLAN = [0, 100, 200, 400, 600, np.inf]                       # tranches du plan d'étude
LABELS_PLAN = ["0-100 m", "100-200 m", "200-400 m", "400-600 m", "600 m et +"]
REF = "200-400 m"                                                   # tranche de référence
BORNES_FINES = {                                                    # pas de 50 m (graphique 1)
    "Paris": list(range(0, 751, 50)),
    "Petite couronne": list(range(0, 1000, 50)) + [1000, 1250, 1500, 1750],
}


def sauver(fig, nom):
    fig.savefig(FIG_DIR / f"{nom}.png", dpi=150, bbox_inches="tight")

---
# Étape 1. Données : sorties du pipeline du projet

Les données ne sont pas traitées dans ce notebook : elles viennent de **deux scripts** du projet, ce qui sépare la *préparation* (reproductible) de l'*analyse* (ce notebook).

| Script | Rôle | Sortie |
|---|---|---|
| `code_dvf.py` | Téléchargement du DVF géolocalisé (Etalab) pour 75, 92, 93, 94 sur 2021-2025, nettoyage | `dvf_appartements_<année>.csv` et `rapport_nettoyage.csv` |
| `stations.py` | Téléchargement des stations IDFM (métro et RER), distance de chaque vente à la station la plus proche, **avec le réseau ouvert à la date de la vente** | `dvf_appartements_stations.parquet` |

**Règles de nettoyage appliquées dans `code_dvf.py`**
1. Ventes simples uniquement (`nature_mutation == "Vente"`) : exclut les VEFA, échanges, adjudications.
2. Une seule disposition par mutation.
3. Exactement **un appartement** par mutation, sans maison ni local commercial (les dépendances sont tolérées et comptées dans `n_dependances`).
4. Prix, surface (9 à 300 m²), coordonnées renseignés et plausibles.
5. Prix au m² entre 1 500 et 30 000 €.
6. Nombre de pièces plafonné à 10.

**Mesure de la distance** : distance à vol d'oiseau, en mètres, après projection en Lambert 93 (EPSG:2154), entre le centre de la parcelle et la station de métro ou de RER la plus proche **ouverte à la date de la vente**.
.

In [2]:
def trouver(nom):
    # Cherche le fichier dans le dossier courant et ses 2 parents, dans les sous-dossiers usuels du projet
    for base in [Path.cwd(), *list(Path.cwd().parents)[:2]]:
        for sous in ["", "data/processed", "data/raw/stations", "data"]:
            p = base / sous / nom
            if p.exists():
                return p
    # Dernier recours : recherche récursive dans le dossier courant uniquement
    trouves = list(Path.cwd().rglob(nom))
    return trouves[0] if trouves else None


chemin = trouver(NOM_PARQUET)
if chemin is None:
    raise FileNotFoundError(
        f"{NOM_PARQUET} introuvable. Lancez d'abord code_dvf.py puis stations.py, "
        "ou indiquez le chemin complet dans la variable `chemin`.")
print("Lecture de :", chemin)
brut = pd.read_parquet(chemin)
print(f"{brut.shape[0]:,} lignes, {brut.shape[1]} colonnes")
brut.head(3)

NameError: name 'NOM_PARQUET' is not defined

In [ ]:
# Rapport de nettoyage produit par code_dvf.py (nombre de mutations restantes à chaque étape)
chemin_rapport = trouver(NOM_RAPPORT)
if chemin_rapport is not None:
    rapport = pd.read_csv(chemin_rapport, index_col=0)
    rapport["Part du brut (%)"] = 100 * rapport["n_mutations"] / rapport["n_mutations"].iloc[0]
    display(rapport)
else:
    print(f"{NOM_RAPPORT} non trouvé : recopier ici le tableau affiché par code_dvf.py.")

In [ ]:
print(apps.shape)
print(apps.groupby(["zone", "annee"]).size().unstack())

NameError: name 'apps' is not defined

### Observations sur le nettoyage

**Lecture des résultats.** Le DVF brut compte 552 046 mutations sur Paris et la petite couronne (départements 75, 92, 93 et 94) pour la période 2021-2025. Après nettoyage, 319 947 ventes d'appartements sont conservées, soit 58 % du volume initial : 143 242 à Paris et 176 705 en petite couronne.

La réduction de l'échantillon se fait en cinq étapes, de poids très inégal :

| Étape | Mutations restantes | Retrait | Part de l'étape précédente |
| --- | ---: | ---: | ---: |
| Mutations brutes | 552 046 | | |
| Ventes simples (hors VEFA, échanges, adjudications) | 491 436 | 60 610 | 11,0 % |
| Une seule disposition par mutation | 488 007 | 3 429 | 0,7 % |
| Un seul appartement par mutation | 330 167 | 157 840 | 32,3 % |
| Variables plausibles (surface, coordonnées) | 327 123 | 3 044 | 0,9 % |
| Prix au m² compris entre 1 500 et 30 000 € | 319 947 | 7 176 | 2,2 % |

Les effectifs annuels par zone sont les suivants :

| Zone | 2021 | 2022 | 2023 | 2024 | 2025 |
| --- | ---: | ---: | ---: | ---: | ---: |
| Paris | 30 427 | 33 138 | 26 861 | 24 460 | 28 356 |
| Petite couronne | 41 244 | 41 669 | 32 013 | 27 911 | 33 868 |

Sur l'ensemble de l'échantillon, le prix médian est de 7 750 €/m² (moyenne de 8 008 €/m²) et la surface médiane de 50 m² (moyenne de 55 m²).

**Cohérence des résultats.** Les volumes parisiens, compris entre 24 000 et 33 000 ventes par an, sont conformes à l'ordre de grandeur d'environ 30 000 ventes d'appartements par an annoncé dans la note de cadrage. L'importance du retrait lors de l'étape « un seul appartement » est attendue : le DVF recense l'ensemble des mutations foncières, y compris les maisons, les terrains et les locaux d'activité, qui n'entrent pas dans le périmètre de l'étude. Les autres filtres retirent chacun moins de 3 % de l'échantillon, ce qui indique que les bornes retenues n'écartent que des cas marginaux.

L'évolution des volumes est cohérente avec le ralentissement du marché immobilier consécutif à la hausse des taux d'intérêt : les ventes diminuent de 2022 à 2024 (-26 % à Paris, -33 % en petite couronne) avant de repartir en 2025 (+16 % et +21 %). La moyenne des prix supérieure à la médiane traduit une distribution asymétrique vers les prix élevés, habituelle sur le marché immobilier.

**Portée statistique.** Avec 24 000 à 42 000 ventes par zone et par année, les médianes globales sont stables et l'aléa d'échantillonnage est négligeable à ce niveau d'agrégation. Il ne le sera plus lors des découpages par tranche de distance ou par commune, dont les effectifs sont examinés à l'étape suivante. L'ampleur des variations de volume entre 2022 et 2024 dépasse celle d'une fluctuation aléatoire : elle relève d'un effet de marché, ou éventuellement d'une différence de couverture des fichiers, qui reste à vérifier par une source externe (statistiques des notaires par exemple). Les statistiques de prix citées ici mélangent les deux zones et les cinq années : elles servent de contrôle de cohérence et ne constituent pas un résultat d'analyse.

**Limites.**
1. *Biais de sélection.* Seules les mutations comportant un unique appartement sont retenues ; les ventes groupées de plusieurs appartements sont exclues. Le filtre sur la nature de mutation écarte par ailleurs les ventes en l'état futur d'achèvement (VEFA), donc le logement neuf vendu sur plan. L'échantillon est ainsi orienté vers l'ancien, ce qui devra être pris en compte pour l'étude des stations ouvertes en 2024, où le neuf est vraisemblablement plus présent.
2. *Étape non décomposée.* Le rapport de nettoyage ne distingue pas les mutations sans appartement des mutations à plusieurs appartements au sein de l'étape « un seul appartement ». Seules les secondes constituent une exclusion au sens strict de l'échantillon ; leur dénombrement permettrait de mesurer précisément le biais décrit ci-dessus.
3. *Bornes de prix.* L'intervalle de 1 500 à 30 000 €/m² est large, en particulier pour Paris, où un prix inférieur à 4 000 €/m² évoque des ventes atypiques (vente partielle, viager, transaction familiale). Les valeurs minimale et maximale observées (1 500 et 30 000 €/m²) coïncident avec les bornes du filtre, ce qui signale la présence de valeurs extrêmes aux limites. Un test de sensibilité avec des bornes propres à chaque zone permettra de vérifier la robustesse des résultats.
4. *Surface.* La borne supérieure de 300 m² autorise quelques très grandes surfaces, qui influencent la moyenne mais peu la médiane, statistique privilégiée dans l'étude.
5. *Prix de mutation.* Le prix d'une mutation inclut les dépendances éventuellement vendues avec l'appartement (variable `n_dependances`), ce qui surestime légèrement le prix au m² de ces ventes.

---
# Étape 2. Mise en forme et contrôles de cohérence

On renomme quelques colonnes pour la lecture, on crée la variable **`zone`** (Paris = département 75, petite couronne = 92, 93, 94) et on vérifie que le fichier est cohérent : colonnes attendues, valeurs manquantes, distances plausibles, doublons.

In [4]:
REQUIS = ["id_mutation", "date_mutation", "annee", "code_departement", "code_commune", "nom_commune",
          "prix_m2", "surface_reelle_bati", "longitude", "latitude", "dist_station_m"]
manquantes = [c for c in REQUIS if c not in brut.columns]
assert not manquantes, f"Colonnes absentes du parquet : {manquantes}"

apps = brut.rename(columns={"surface_reelle_bati": "surface",
                            "nombre_pieces_principales": "pieces",
                            "dist_station_m": "dist_station"}).copy()
apps["date_mutation"] = pd.to_datetime(apps["date_mutation"])
apps["annee"] = apps["date_mutation"].dt.year
apps["code_departement"] = apps["code_departement"].astype(str).str.zfill(2)
apps["code_commune"] = apps["code_commune"].astype(str)
apps["zone"] = np.where(apps["code_departement"] == "75", "Paris", "Petite couronne")
apps = apps[apps["annee"].isin(ANNEES)].dropna(subset=["dist_station", "prix_m2"]).reset_index(drop=True)

print(f"{len(apps):,} ventes d'appartements retenues pour l'analyse")
print("Départements présents :", sorted(apps.code_departement.unique()))
print("Doublons d'id_mutation :", apps.id_mutation.duplicated().sum())
print("\nValeurs manquantes (%) sur les colonnes clés :")
display((100 * apps[["prix_m2", "surface", "pieces", "dist_station", "latitude", "longitude"]].isna().mean()).round(2))
print("Distance à la station (m) par zone :")
display(apps.groupby("zone")["dist_station"].describe(percentiles=[.5, .9, .99]).round(0))

NameError: name 'brut' is not defined

In [ ]:
# Contrôle du réseau daté : la colonne date_ouverture_station est-elle une vraie date d'ouverture ?
if "date_ouverture_station" in apps.columns:
    d = pd.to_datetime(apps["date_ouverture_station"], errors="coerce")
    print("Dates d'ouverture les plus fréquentes parmi les stations les plus proches des ventes :")
    display(d.dt.date.value_counts().head(10))
    print("Ventes dont la station la plus proche a ouvert depuis 2021 :", int((d >= "2021-01-01").sum()))
    print("Dates d'ouverture manquantes :", int(d.isna().sum()))

---
# Étape 3. Construction des variables d'analyse

- **`tranche`** : les cinq tranches du plan d'étude (0-100, 100-200, 200-400, 400-600, 600+). Elles diffèrent de `classe_dist` du pipeline (pas de 100 m jusqu'à 500+) : on recalcule à partir de la distance continue.
- **`prix_rel`** : prix au m² **divisé par la médiane de sa zone et de son année**. Cette standardisation simple permet de **regrouper plusieurs années** sans que la baisse générale des prix ne fausse les comparaisons. C'est une division, pas un modèle.

**Limites à garder en tête**
- Distance à vol d'oiseau, pas à pied.
- Les coordonnées DVF sont celles du **centre de la parcelle**, pas de l'immeuble : l'incertitude de position est de quelques dizaines de mètres, ce qui limite la lecture fine à 50 m près.

In [ ]:
apps["tranche"] = pd.cut(apps["dist_station"], BORNES_PLAN, labels=LABELS_PLAN, right=False)
apps["prix_rel"] = apps["prix_m2"] / apps.groupby(["zone", "annee"])["prix_m2"].transform("median")
apps[["zone", "nom_commune", "annee", "surface", "prix_m2", "dist_station", "tranche", "prix_rel"]].head()

---
# Étape 4. Contrôle de qualité : effectifs et distributions

Avant d'interpréter la moindre courbe, on vérifie **combien de ventes** il y a par année, par zone et par tranche de distance, et à quoi ressemble la distribution des prix.

Pourquoi c'est indispensable : une médiane calculée sur 15 ventes est instable. Les zigzags d'une courbe correspondent très souvent à de petits effectifs, pas à un phénomène réel.

In [ ]:
print("Nombre de ventes par zone et par année")
display(apps.pivot_table(index="zone", columns="annee", values="prix_m2", aggfunc="size"))

print("Prix médian au m² (€) par zone et par année")
display(apps.pivot_table(index="zone", columns="annee", values="prix_m2", aggfunc="median"))

print("Nombre de ventes par tranche de distance (toutes années)")
display(apps.pivot_table(index="tranche", columns="zone", values="prix_m2", aggfunc="size", observed=True))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for zone, c in zip(ZONES, ["#1f5fbf", "#e8643c"]):
    axes[0].hist(apps.loc[apps.zone == zone, "prix_m2"], bins=60, alpha=.6, label=zone, color=c)
axes[0].set(title="Distribution du prix au m²", xlabel="€/m²", ylabel="Nombre de ventes"); axes[0].legend()
for zone, c in zip(ZONES, ["#1f5fbf", "#e8643c"]):
    axes[1].hist(apps.loc[(apps.zone == zone) & (apps.dist_station < 2000), "dist_station"], bins=60, alpha=.6, label=zone, color=c)
axes[1].set(title="Distance à la station la plus proche", xlabel="m", ylabel="Nombre de ventes"); axes[1].legend()
apps.boxplot(column="prix_m2", by=["zone", "annee"], ax=axes[2], rot=60, grid=False)
axes[2].set(title="Prix au m² par zone et année", xlabel="", ylabel="€/m²"); fig.suptitle("")
plt.tight_layout(); sauver(fig, "00_qualite"); plt.show()

---
# Étape 5. Géographie des prix et position des stations

**Idée.** Le risque principal du projet est l'**effet de localisation** : à Paris, être proche d'une station va souvent de pair avec être central, donc cher. On le vérifie visuellement avant de mesurer quoi que ce soit.

Trois lectures :
1. Prix médian par arrondissement (Paris) et par commune (petite couronne).
2. Carte de chaleur du prix médian avec les stations en surimpression.
3. Lien entre le prix médian d'une commune et la part de ses ventes situées à moins de 300 m d'une station.

In [ ]:
d = apps[apps.annee == ANNEES[-1]]
par_commune = (d.groupby(["zone", "code_commune", "nom_commune"])
                 .agg(prix_median=("prix_m2", "median"), n=("prix_m2", "size")).reset_index())
par_commune = par_commune[par_commune.n >= MIN_N]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
p = par_commune[par_commune.zone == "Paris"].sort_values("prix_median")
axes[0].barh(p.nom_commune.str.replace("Paris ", "").str.replace(" Arrondissement", ""), p.prix_median, color="#1f5fbf")
axes[0].set(title=f"Paris : prix médian par arrondissement ({ANNEES[-1]})", xlabel="€/m²")
c = par_commune[par_commune.zone == "Petite couronne"].sort_values("prix_median")
c = pd.concat([c.head(10), c.tail(10)])
axes[1].barh(c.nom_commune, c.prix_median, color="#e8643c")
axes[1].set(title=f"Petite couronne : 10 communes les moins / plus chères ({ANNEES[-1]})", xlabel="€/m²")
plt.tight_layout(); sauver(fig, "01_prix_par_commune"); plt.show()

In [ ]:
st_ll = None
chemin_st = trouver(NOM_GEOJSON_STATIONS)
if chemin_st is not None:
    st = gpd.read_file(chemin_st)
    if "mode" in st.columns:
        st = st[st["mode"].isin(["METRO", "RER"])]
    st_ll = st[st.geometry.geom_type == "Point"].to_crs(4326)
sel = apps[(apps.longitude.between(2.20, 2.50)) & (apps.latitude.between(48.76, 48.96))]
fig, ax = plt.subplots(figsize=(9, 8))
hb = ax.hexbin(sel.longitude, sel.latitude, C=sel.prix_m2, reduce_C_function=np.median,
               gridsize=55, mincnt=10, cmap="YlOrRd")
if st_ll is not None:
    ax.scatter(st_ll.geometry.x, st_ll.geometry.y, s=6, c="black", alpha=.6, label="Stations")
ax.set_xlim(2.20, 2.50); ax.set_ylim(48.76, 48.96); ax.set_aspect(1 / np.cos(np.radians(48.86)))
plt.colorbar(hb, label="Prix médian au m² (€)")
if st_ll is not None:
    ax.legend(loc="lower left")
ax.set(title="Carte de chaleur du prix au m² (toutes années) et stations", xlabel="Longitude", ylabel="Latitude")
sauver(fig, "02_carte_prix_stations"); plt.show()

In [ ]:
loc = (apps.assign(proche=apps.dist_station < 300)
           .groupby(["zone", "code_commune"])
           .agg(prix_median=("prix_m2", "median"), part_proche=("proche", "mean"), n=("prix_m2", "size"))
           .reset_index())
loc = loc[loc.n >= MIN_N]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
for ax, zone in zip(axes, ZONES):
    l = loc[loc.zone == zone]
    ax.scatter(100 * l.part_proche, l.prix_median, s=np.sqrt(l.n) * 2, alpha=.6)
    r = l["part_proche"].corr(l["prix_median"])      # coefficient de corrélation descriptif, pas un modèle
    ax.set(title=f"{zone} (corrélation de Pearson = {r:.2f}, {len(l)} communes)",
           xlabel="Part des ventes à moins de 300 m d'une station (%)", ylabel="Prix médian au m² (€)")
plt.tight_layout(); sauver(fig, "03_localisation_vs_stations"); plt.show()

### Lecture initiale à confirmer avec vos graphiques
- Si les arrondissements/communes les plus chers sont aussi ceux où une grande part des ventes est près d'une station, alors **une partie de l'écart brut vient de la localisation** : les stations sont dans les zones déjà chères et denses.
- C'est exactement ce que cherche à isoler la comparaison « à commune égale » de l'étape 8.
- Une corrélation sur quelques dizaines de communes décrit un lien, elle ne prouve rien sur la cause. Attention à la taille des bulles : les petites communes pèsent peu.

### ✍️ Observations sur la géographie (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 6. Courbe de décote : prix médian selon la distance à la station

**Objectif.** Visualiser comment le prix au m² évolue quand on s'éloigne de la station, année par année.

**Construction.** Tranches de 50 m (puis plus larges en petite couronne au-delà de 1 000 m où il y a peu de ventes). Médiane par tranche et par année. **Les points dont l'effectif est inférieur à 30 ventes sont masqués.**

On produit aussi une version **lissée** (moyenne mobile sur 3 tranches voisines), uniquement pour aider la lecture : c'est de la présentation, pas un modèle.

In [ ]:
def courbe_distance(d, bornes, min_n=MIN_N, lisse=False):
    d = d[(d.dist_station >= bornes[0]) & (d.dist_station < bornes[-1])]
    tr = pd.cut(d.dist_station, bornes, right=False)
    g = d.groupby(tr, observed=False)["prix_m2"].agg(mediane="median", n="size")   # observed=False : garde les tranches vides
    g["milieu"] = [iv.mid for iv in g.index]
    g.loc[g.n < min_n, "mediane"] = np.nan
    if lisse:
        g["mediane"] = g["mediane"].rolling(3, center=True, min_periods=2).mean()
    return g


def tracer_courbes(lisse, nom):
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
    couleurs = plt.cm.Blues(np.linspace(.45, 1, len(ANNEES)))
    for ax, zone in zip(axes, ZONES):
        for col, a in zip(couleurs, ANNEES):
            g = courbe_distance(apps[(apps.zone == zone) & (apps.annee == a)], BORNES_FINES[zone], lisse=lisse)
            ax.plot(g.milieu, g.mediane, color=col, lw=2, label=a)
        ax.set(title=zone, xlabel="Distance à la station la plus proche (m)", ylabel="Prix médian (€/m²)")
        ax.legend(title="Année", frameon=False)
    fig.suptitle("Prix au m² selon la distance au métro/RER, par année" + (" (lissé)" if lisse else ""), x=.02, ha="left")
    plt.tight_layout(); sauver(fig, nom); plt.show()


tracer_courbes(lisse=False, nom="04_courbe_distance")
tracer_courbes(lisse=True, nom="04b_courbe_distance_lissee")

In [ ]:
# Effectifs par tranche de 50 m (toutes années) : c'est ce qui conditionne la fiabilité des courbes
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6))
for ax, zone in zip(axes, ZONES):
    g = courbe_distance(apps[apps.zone == zone], BORNES_FINES[zone], min_n=0)
    largeur = np.diff(BORNES_FINES[zone])
    ax.bar(g.milieu, g.n, width=largeur * .9, color="#6a9bd8")
    ax.set(title=f"{zone} : nombre de ventes par tranche (toutes années)", xlabel="Distance (m)", ylabel="Ventes")
plt.tight_layout(); sauver(fig, "05_effectifs_distance"); plt.show()

### Lecture initiale (d'après vos premiers résultats, à confirmer avec cette exécution)
- **Paris** : prix quasi plat jusqu'à 200-250 m (maximum vers 175-225 m) puis décroissance nette au-delà de 450 m. L'écart entre 0 m et 700 m est d'environ 1 000 à 1 500 €/m². Toutes les courbes descendent d'une année à l'autre.
- **Petite couronne** : décroissance forte et continue (environ 6 500-7 000 €/m² près des stations contre 4 700-5 000 €/m² vers 1 500 m), puis stabilisation au-delà de 1 000 m.
- **Logique ?** Oui : le réseau est dense à Paris (presque tout est à moins de 700 m d'une station, d'où un effet limité) et rare en banlieue (d'où un effet plus marqué).
- **Significatif ?** Prudence en petite couronne : les zigzags de 300 à 500 €/m² d'un point à l'autre sont typiques de petits effectifs. Se référer au graphique des effectifs et à la version lissée.
- **Piège** : cette courbe mélange des communes différentes. Une commune proche de Paris cumule stations nombreuses **et** prix élevés. Cette courbe **ne peut donc pas isoler** l'effet de la station (voir étape 8).

### ✍️ Observations sur la courbe de décote (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 7. Évolution des prix 2021-2025 selon la distance (indice base 100 = 2021)

**Objectif.** Savoir si la baisse des prix (retournement du marché après la hausse des taux) touche de la même façon les biens proches et éloignés des stations.

**Construction.** Prix médian par zone, par tranche de distance **du plan d'étude** et par année, ramené à 100 en 2021. On affiche aussi les **effectifs**, car un indice est très sensible à la composition de l'année de base.

In [ ]:
t = apps.pivot_table(index=["zone", "tranche"], columns="annee", values="prix_m2", aggfunc="median", observed=True)
n = apps.pivot_table(index=["zone", "tranche"], columns="annee", values="prix_m2", aggfunc="size", observed=True)
indice = t.div(t[ANNEES[0]], axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
couleurs = plt.cm.Blues(np.linspace(.35, 1, len(LABELS_PLAN)))[::-1]
for ax, zone in zip(axes, ZONES):
    for col, lab in zip(couleurs, LABELS_PLAN):
        if (zone, lab) in indice.index:
            ax.plot(ANNEES, indice.loc[(zone, lab)].values, marker="o", lw=2, color=col, label=lab)
    ax.axhline(100, color="grey", lw=.8)
    ax.set(title=zone, ylabel="Indice prix médian (base 100 = 2021)"); ax.set_xticks(ANNEES)
    ax.legend(title="Distance", frameon=False)
fig.suptitle("Évolution des prix selon la distance à la station", x=.02, ha="left")
plt.tight_layout(); sauver(fig, "06_indice_base100"); plt.show()

print("Effectifs par zone, tranche et année"); display(n)
print("Indice (base 100 = 2021)"); display(indice)

### Lecture initiale (à confirmer)
- **Paris** : baisse générale d'environ 10 à 13 % entre 2021 et 2024, puis léger rebond en 2025 pour les tranches proches. Les courbes sont très proches (écart de 2 à 4 points) : **on ne peut pas conclure à un effet différencié de la distance**.
- **Petite couronne** : légère hausse en 2022, puis chute jusqu'en 2024. En 2025, les biens éloignés rebondissent davantage que les biens proches. C'est **contre-intuitif** (on attendrait l'inverse) et doit être traité comme une piste, pas comme un fait.
- **Prudence** : un indice base 100 dépend de l'année 2021 ; si la composition des ventes de 2021 était particulière (peu de ventes, type de biens différent), tout l'indice est décalé. D'où l'importance du tableau d'effectifs.

### ✍️ Observations sur l'évolution des prix (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 8. Prime de proximité : brute et « à commune égale »

**Objectif.** Mesurer l'écart de prix entre les biens **proches** (moins de 300 m) et **éloignés** (600 m et plus), et voir **quelle part de cet écart disparaît quand on compare des biens de la même commune**.

**Définitions (sans modèle).**
- **Prime brute** = médiane(prix/m² des biens à moins de 300 m) ÷ médiane(prix/m² des biens à 600 m et plus) − 1, sur toute la zone.
- **Prime à commune égale** : dans chaque commune (arrondissement à Paris), on calcule le même écart, puis on fait la **moyenne pondérée** des écarts, avec comme poids le plus petit des deux effectifs (proches / éloignés). On ne garde que les communes avec au moins 5 ventes dans chaque groupe.
- **Intervalle de confiance** : 200 rééchantillonnages bootstrap des ventes (percentiles 2,5 % et 97,5 %). Il traduit l'incertitude d'échantillonnage, pas les biais éventuels.

> Si vos graphiques d'origine utilisaient une pondération différente, les valeurs peuvent légèrement différer : adapter la fonction `prime_proximite` à votre code initial.

In [ ]:
def prime_proximite(d, proche=300, loin=600, maille="code_commune", min_n=5):
    grp = np.where(d.dist_station < proche, "proche", np.where(d.dist_station >= loin, "loin", "milieu"))
    d = d.assign(grp=grp)
    d = d[d.grp != "milieu"]
    med = d.groupby("grp")["prix_m2"].median()
    if not {"proche", "loin"} <= set(med.index):
        return np.nan, np.nan, 0
    brute = 100 * (med["proche"] / med["loin"] - 1)
    g = d.groupby([maille, "grp"])["prix_m2"].agg(["median", "size"]).unstack("grp").dropna()
    g = g[(g[("size", "proche")] >= min_n) & (g[("size", "loin")] >= min_n)]
    if len(g) == 0:
        return brute, np.nan, 0
    ecart = g[("median", "proche")] / g[("median", "loin")] - 1
    poids = np.minimum(g[("size", "proche")], g[("size", "loin")])
    return brute, 100 * np.average(ecart, weights=poids), len(g)


lignes = []
for zone in ZONES:
    for a in ANNEES:
        d = apps[(apps.zone == zone) & (apps.annee == a)]
        b, c, k = prime_proximite(d)
        bs = []
        for _ in range(N_BOOT):
            ech = d.sample(frac=1, replace=True, random_state=int(rng.integers(1e9)))
            bs.append(prime_proximite(ech)[:2])
        bs = np.array(bs, dtype=float)
        lignes.append(dict(zone=zone, annee=a, brute=b, commune_egale=c, n_communes=k, n_ventes=len(d),
                           brute_lo=np.nanpercentile(bs[:, 0], 2.5), brute_hi=np.nanpercentile(bs[:, 0], 97.5),
                           ce_lo=np.nanpercentile(bs[:, 1], 2.5), ce_hi=np.nanpercentile(bs[:, 1], 97.5)))
prime = pd.DataFrame(lignes)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, zone in zip(axes, ZONES):
    p = prime[prime.zone == zone]
    ax.plot(p.annee, p.brute, marker="o", lw=2, color="#e8643c", label="Brute")
    ax.fill_between(p.annee, p.brute_lo, p.brute_hi, color="#e8643c", alpha=.15)
    ax.plot(p.annee, p.commune_egale, marker="o", lw=2, color="#1f5fbf", label="À commune égale")
    ax.fill_between(p.annee, p.ce_lo, p.ce_hi, color="#1f5fbf", alpha=.15)
    ax.axhline(0, color="grey", lw=.8); ax.set_xticks(ANNEES)
    ax.set(title=zone, ylabel="Écart de prix < 300 m vs ≥ 600 m (%)"); ax.legend(frameon=False)
fig.suptitle("Prime de proximité : brute et à commune égale (bandes = IC 95 % bootstrap)", x=.02, ha="left")
plt.tight_layout(); sauver(fig, "07_prime_proximite"); plt.show()
prime.round(1)

### Lecture initiale (à confirmer)
- **Paris** : prime brute d'environ 13 % (2021) à 17 % (2025) ; à commune égale, environ 8 % en 2021 puis 4-5 % en 2023-2024, remontant vers 7 % en 2025.
- **Petite couronne** : prime brute très élevée (environ 29 % puis 22 %), mais à commune égale seulement 7-9 %, **stable**.
- **C'est le résultat central du projet** : **plus de la moitié de l'écart brut est un effet de localisation** (les communes bien desservies sont aussi les plus chères), pas un effet propre de la distance à la station. L'effet « net » de proximité est de l'ordre de 5 à 9 %.
- **La baisse de la prime brute en petite couronne** n'apparaît pas à commune égale : elle vient donc d'un changement *entre* communes, pas de la valeur de la proximité.
- **Lecture des bandes** : si les bandes de deux années se chevauchent largement, la variation entre ces deux années n'est pas distinguable du hasard d'échantillonnage.

**Limites**
- La commune reste une maille grossière : un quartier proche de la gare peut différer du reste de la commune (passage à l'IRIS prévu).
- Les seuils 300 m / 600 m sont un choix : voir l'étape de robustesse.

### ✍️ Observations sur la prime de proximité (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 9. Tranches du plan d'étude et test descriptif de H2

On revient aux **tranches prévues dans le plan** (0-100, 100-200, 200-400, 400-600, 600+) pour examiner H2 : y a-t-il une décote à moins de 100 m par rapport à 200-400 m ? Où se situe le maximum ?

On utilise `prix_rel` (prix divisé par la médiane de la zone et de l'année) pour pouvoir **regrouper les cinq années**.

**Deux lectures**
- **(a) Brute** : médiane de `prix_rel` par tranche, avec IC bootstrap.
- **(b) À commune égale** : dans chaque commune, écart de la médiane de chaque tranche par rapport à la tranche de référence 200-400 m, puis moyenne pondérée (poids = le plus petit des deux effectifs). On ne garde que les communes avec au moins 5 ventes dans la tranche et dans la référence.

Si l'intervalle de confiance de la tranche 0-100 m **exclut** 0 vers le bas, la décote de H2 est plausible. S'il contient 0, on ne peut pas trancher.

In [ ]:
def profil_brut(d):
    return d.groupby("tranche", observed=True)["prix_rel"].median().reindex(LABELS_PLAN)


def profil_intra(d, ref=REF, maille="code_commune", min_n=5):
    g = d.groupby([maille, "tranche"], observed=True)["prix_rel"].agg(med="median", n="size").reset_index()
    r = g[g.tranche == ref].set_index(maille)[["med", "n"]].rename(columns={"med": "med_ref", "n": "n_ref"})
    g = g.merge(r, left_on=maille, right_index=True)
    g = g[(g.n >= min_n) & (g.n_ref >= min_n)]
    g["ecart"] = g.med / g.med_ref - 1
    g["w"] = np.minimum(g.n, g.n_ref)
    num = (g.ecart * g.w).groupby(g.tranche, observed=True).sum()
    den = g.w.groupby(g.tranche, observed=True).sum()
    return (100 * num / den).reindex(LABELS_PLAN)


res = {}
for zone in ZONES:
    d = apps[apps.zone == zone].dropna(subset=["tranche"])
    brut_pt, intra_pt = profil_brut(d), profil_intra(d)
    bb, bi = [], []
    for _ in range(N_BOOT):
        ech = d.sample(frac=1, replace=True, random_state=int(rng.integers(1e9)))
        bb.append(profil_brut(ech).values); bi.append(profil_intra(ech).values)
    bb, bi = np.array(bb, dtype=float), np.array(bi, dtype=float)
    res[zone] = pd.DataFrame({
        "n": d.groupby("tranche", observed=True).size().reindex(LABELS_PLAN),
        "prix_rel_median": brut_pt.values, "brut_lo": np.nanpercentile(bb, 2.5, axis=0), "brut_hi": np.nanpercentile(bb, 97.5, axis=0),
        "ecart_vs_ref_%": intra_pt.values, "intra_lo": np.nanpercentile(bi, 2.5, axis=0), "intra_hi": np.nanpercentile(bi, 97.5, axis=0)},
        index=LABELS_PLAN)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for j, zone in enumerate(ZONES):
    r = res[zone]; x = np.arange(len(LABELS_PLAN))
    ax = axes[0, j]
    ax.errorbar(x, r.prix_rel_median, yerr=[r.prix_rel_median - r.brut_lo, r.brut_hi - r.prix_rel_median],
                fmt="o-", capsize=4, color="#e8643c", lw=2)
    ax.set_xticks(x); ax.set_xticklabels(LABELS_PLAN); ax.axhline(1, color="grey", lw=.8)
    ax.set(title=f"{zone} : prix relatif médian (brut)", ylabel="Prix / médiane de la zone et de l'année")
    ax = axes[1, j]
    ax.errorbar(x, r["ecart_vs_ref_%"], yerr=[r["ecart_vs_ref_%"] - r.intra_lo, r.intra_hi - r["ecart_vs_ref_%"]],
                fmt="o-", capsize=4, color="#1f5fbf", lw=2)
    ax.set_xticks(x); ax.set_xticklabels(LABELS_PLAN); ax.axhline(0, color="grey", lw=.8)
    ax.set(title=f"{zone} : écart à la tranche {REF}, à commune égale", ylabel="Écart (%)")
fig.suptitle("Courbe de décote par tranches du plan (IC 95 % bootstrap)", x=.02, ha="left")
plt.tight_layout(); sauver(fig, "08_tranches_plan_H2"); plt.show()
for zone in ZONES:
    print(zone); display(res[zone].round(3))

### Lecture initiale (à confirmer)
- Au vu de vos premiers résultats, **H2 est plausible à Paris** (prix plus bas à 0-100 m, maximum vers 175-225 m) mais l'amplitude est faible (environ 1 à 3 %). La conclusion dépend de l'intervalle de confiance : s'il chevauche 0, la décote n'est **pas démontrée**.
- **Intervalle « optimal »** : s'il existe, c'est la ou les tranches dont l'écart est maximal **et** dont l'IC ne chevauche pas celui des tranches voisines. Sinon, la conclusion honnête est : « pas de distance optimale nettement identifiable à cette précision ».
- **Précision de position** : les coordonnées sont celles du centre de la parcelle ; distinguer 0-100 m de 100-200 m est donc fragile. À mentionner systématiquement.

### ✍️ Observations sur H1 et H2 (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 10. Surface et prix au m² (H3) et effet de composition

**H3** : le prix au m² décroît avec la surface, plus fortement pour les petits logements.

**Pourquoi c'est aussi une question de méthode.** Si les petits logements sont plus nombreux près des stations (studios en centre, bien desservi) **et** plus chers au m², alors une partie de la « prime de proximité » viendrait de la **composition** des ventes (plus de petites surfaces), pas de la station elle-même. On regarde donc :
1. Le prix relatif médian par tranche de surface.
2. La répartition des surfaces selon la distance à la station.

In [ ]:
BORNES_SURF = [0, 20, 30, 40, 60, 80, 120, np.inf]
LABELS_SURF = ["< 20", "20-30", "30-40", "40-60", "60-80", "80-120", "120 et +"]
apps["tranche_surface"] = pd.cut(apps["surface"], BORNES_SURF, labels=LABELS_SURF, right=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for j, zone in enumerate(ZONES):
    d = apps[apps.zone == zone]
    g = d.groupby("tranche_surface", observed=True).agg(prix_rel=("prix_rel", "median"), n=("prix_rel", "size"))
    axes[0, j].plot(g.index.astype(str), g.prix_rel, "o-", lw=2, color="#1f5fbf")
    axes[0, j].axhline(1, color="grey", lw=.8)
    axes[0, j].set(title=f"{zone} : prix relatif médian selon la surface", ylabel="Prix / médiane zone-année", xlabel="Surface (m²)")
    ax2 = axes[0, j].twinx(); ax2.bar(g.index.astype(str), g.n, alpha=.2, color="grey"); ax2.set_ylabel("Ventes"); ax2.grid(False)
    comp = pd.crosstab(d.tranche, d.tranche_surface, normalize="index") * 100
    comp.plot(kind="bar", stacked=True, ax=axes[1, j], colormap="Blues", width=.8, legend=(j == 1))
    axes[1, j].set(title=f"{zone} : répartition des surfaces selon la distance", xlabel="Tranche de distance", ylabel="% des ventes")
    axes[1, j].tick_params(axis="x", rotation=0)
plt.tight_layout(); sauver(fig, "09_surface_H3"); plt.show()

print("Surface médiane (m²) par tranche de distance")
display(apps.pivot_table(index="tranche", columns="zone", values="surface", aggfunc="median", observed=True))

### Lecture initiale (à confirmer)
- **H3** : si la courbe est décroissante avec un pic marqué pour les petites surfaces, H3 est confirmée de façon descriptive.
- **Composition** : si la part de petites surfaces change nettement selon la tranche de distance, alors les courbes des étapes 6 à 9 sont en partie un effet de composition. Prochaine étape naturelle : refaire les graphiques **par tranche de surface** (voir pistes).

### ✍️ Observations sur la surface (H3) (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 11. Robustesse : le résultat dépend-il des seuils ?

La prime de l'étape 8 dépend du choix « moins de 300 m » contre « 600 m et plus ». On refait le calcul à commune égale avec d'autres seuils. Si les trois courbes racontent la même histoire, le résultat est robuste ; sinon, il faut en parler dans les limites.

In [ ]:
SEUILS = [(250, 500), (300, 600), (200, 800), (150, 400)]
rob = []
for zone in ZONES:
    for a in ANNEES:
        d = apps[(apps.zone == zone) & (apps.annee == a)]
        for (p, l) in SEUILS:
            b, c, k = prime_proximite(d, proche=p, loin=l)
            rob.append(dict(zone=zone, annee=a, seuils=f"< {p} m vs ≥ {l} m", brute=b, commune_egale=c))
rob = pd.DataFrame(rob)

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for j, zone in enumerate(ZONES):
    for lab, g in rob[rob.zone == zone].groupby("seuils"):
        axes[0, j].plot(g.annee, g.brute, marker="o", label=lab)
        axes[1, j].plot(g.annee, g.commune_egale, marker="o", label=lab)
    axes[0, j].set(title=f"{zone} : prime brute (%)"); axes[1, j].set(title=f"{zone} : prime à commune égale (%)")
    axes[1, j].set_xticks(ANNEES); axes[0, j].legend(frameon=False, fontsize=8)
plt.tight_layout(); sauver(fig, "10_robustesse_seuils"); plt.show()
rob.pivot_table(index=["zone", "seuils"], columns="annee", values="commune_egale").round(1)

### ✍️ Observations sur la robustesse (à compléter après exécution)
- **Ce que je lis** : …
- **Est-ce logique ?** : …
- **Est-ce significatif ?** (effectifs, intervalles, hasard ?) : …
- **Remarques / limites** : …

---
# Étape 12. Synthèse

| Hypothèse | Verdict provisoire | Éléments à l'appui | Réserves |
|---|---|---|---|
| **H1** : le prix baisse avec la distance, à quartier comparable | *à compléter* | Étapes 6, 8, 9 | Maille = commune ; distance à vol d'oiseau ; effet de composition (étape 10) |
| **H2** : décote à moins de 100 m par rapport à 200-400 m | *à compléter* | Étape 9 (IC bootstrap) | Position = centre de parcelle ; amplitude faible ; IC à regarder |
| **H3** : le prix au m² décroît avec la surface, surtout pour les petits logements | *à compléter* | Étape 10 | Surface et nombre de pièces corrélés |

## Ce que nos résultats permettent déjà d'affirmer (à ajuster selon l'exécution)
1. L'**écart brut** de prix entre proche et éloigné est important, mais **une grande part vient de la localisation** : à commune égale, la prime de proximité est nettement plus faible.
2. La **baisse des prix 2021-2025** touche toutes les distances ; les différences entre tranches sont soit faibles (Paris), soit à interpréter avec prudence (petite couronne).
3. La question de la **distance optimale** n'a de réponse que si les intervalles de confiance de l'étape 9 permettent de départager les tranches.

## Limites générales (à reprendre dans le rapport)
- DVF : ni étage, ni année de construction, ni balcon ; nombre de pièces peu fiable.
- Coordonnées au centre de la parcelle ; distance à vol d'oiseau.
- Prix de mutation incluant d'éventuelles dépendances.
- Comparaison à la commune : encore grossière (IRIS prévu).
- Les IC bootstrap ne tiennent pas compte de la corrélation spatiale entre ventes voisines : ils sont **optimistes**.
- Aucune causalité n'est affirmée : on décrit des écarts de prix observés, pas l'effet d'une politique ou d'une mesure.

## Pistes pour l'étape suivante (toujours sans modèle)
- Descendre de la commune à l'**IRIS** (jointure spatiale avec les contours IGN/INSEE).
- Refaire les étapes 6 à 9 **par tranche de surface** pour neutraliser l'effet de composition.
- **Distance au grand axe routier et au parc** (OpenStreetMap) : tester si la décote à moins de 100 m disparaît loin des axes bruyants.
- **Type de station** (métro, RER, tram) et **fréquentation RATP** (terciles) : la prime varie-t-elle ?
- **Avant/après juin 2024** (prolongements des lignes 11 et 14) : comparer les biens proches des nouvelles stations à des zones témoins, de façon descriptive.
- Appariement descriptif de paires de ventes comparables (même commune, surface proche, distances différentes).